Imports

In [1]:
import requests
import pandas as pd
import time



help function for building the right URL and returns the response as a JSON    

In [2]:
TMDB_API_KEY = "381056fa40bc22a39a9b6c16f58a2465"
BASE_URL = "https://api.themoviedb.org/3"
def tmdb_get(endpoint, params=None, timeout=10):
    if params is None:
        params = {}
    params["api_key"] = TMDB_API_KEY

    url = BASE_URL + endpoint
    r = requests.get(url, params = params, timeout = timeout)
    r.raise_for_status()
    r.status_code
    return r.json()



function that collects movies by iterating through several pages
the movies that are collectet are stored in a list an converted into a Data Frame

In [3]:
def collect_movies(year=2010, pages=3, sort_by="popularity.desc", language="en-US", sleep=0.2):
    all_movies = []

    for page in range(1, pages+1):
        data = tmdb_get(
            "/discover/movie",
            params={
                "primary_release_year": year,
                "sort_by": sort_by,
                "page": page,
                "language": language,
            }
        )
        all_movies.extend(data["results"])
        time.sleep(0.2)
    return pd.DataFrame(all_movies)

collect the movies in a Data Frame

In [4]:
years = range(1990,2025)
frames = []

for y in years:
    frames.append(collect_movies(y,pages=5))

df_movies = pd.concat(frames, ignore_index=True).drop_duplicates("id")

function to get additional movie details

In [5]:
def get_movie_details(movie_id):

    data = tmdb_get(f"/movie/{movie_id}")

    return {
        "id": data.get("id"),
        "budget": data.get("budget"),
        "revenue": data.get("revenue"),
        "runtime": data.get("runtime"),
        "genres": [g["name"] for g in data.get("genres", [])]
    }

goes through all collected movie id´s and request the additional details. 
the results are stored in a list

In [6]:
details = []

for movie_id in df_movies["id"]:

    try:
        d = get_movie_details(movie_id)
        details.append(d)

    except:
        continue

    time.sleep(0.2)

convert collected movie details into a pandas DataFrame

In [7]:
df_details = pd.DataFrame(details)


merge the basic movie data with the detailed movie data unsing the movie id

In [8]:
df_full = df_movies.merge(df_details, on ="id", how="left")


In [9]:
df_full.to_excel("movies_datassset.xlsx", index=False)